---

# 🧪 AZURE DATABRICKS – FINAL ASSESSMENT



## 🎯 Objective

Build a complete data pipeline using:
**PySpark + SQL + Delta Lake + Incremental Load + DLT + Unity Catalog**

---

## 🔷 PART 1 — DATAFRAME (PYSPARK)

### Dataset (Given)

```python
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]

columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
```

### Tasks

1. Create DataFrame
2. Add required derived columns
3. Filter high-value patients
4. Perform aggregation by department
5. Sort results appropriately

---


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]

columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
df=spark.createDataFrame(data,columns)

In [0]:
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
df=df.withColumn("bill_amount",df.consultation_fee*df.tests_count)
df.show()

+--------+------------+---------+-----------+----------------+-----------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|bill_amount|
+--------+------------+---------+-----------+----------------+-----------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|       5000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|       6000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|       1500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      10000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|       7000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|       9000|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|       5000|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|       3000|
+--------+------------+---------+----------

In [0]:
df.filter(df.bill_amount>5000).show()

+--------+------------+---------+-----------+----------------+-----------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|bill_amount|
+--------+------------+---------+-----------+----------------+-----------+-----------+
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|       6000|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      10000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|       7000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|       9000|
+--------+------------+---------+-----------+----------------+-----------+-----------+



In [0]:
df.groupBy(df.department).count().show()

+-----------+-----+
| department|count|
+-----------+-----+
| Cardiology|    3|
|Orthopedics|    2|
|Dermatology|    2|
|  Neurology|    1|
+-----------+-----+



In [0]:
df.orderBy(df.bill_amount.desc()).show()

+--------+------------+---------+-----------+----------------+-----------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|bill_amount|
+--------+------------+---------+-----------+----------------+-----------+-----------+
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      10000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|       9000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|       7000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|       6000|
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|       5000|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|       5000|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|       3000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|       1500|
+--------+------------+---------+----------

---

## 🔷 PART 2 — SPARK SQL

### Tasks

1. Convert DataFrame to a temp view
2. Write SQL to:

   * Fetch specific department records
   * Calculate revenue per city
   * Identify top patients
   * Count patients per department

---

In [0]:
df.createOrReplaceTempView("patients")

In [0]:
spark.sql("select * from patients").show()

+--------+------------+---------+-----------+----------------+-----------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|bill_amount|
+--------+------------+---------+-----------+----------------+-----------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|       5000|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|       6000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|       1500|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      10000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|       7000|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|       9000|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|       5000|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|       3000|
+--------+------------+---------+----------

In [0]:
spark.sql("select * from patients where department='Cardiology'").show()

+--------+------------+---------+----------+----------------+-----------+-----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|bill_amount|
+--------+------------+---------+----------+----------------+-----------+-----------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|       5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      10000|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|       5000|
+--------+------------+---------+----------+----------------+-----------+-----------+



In [0]:
spark.sql("select city ,sum(bill_amount) Revenue from patients group by city").show()

+---------+-------+
|     city|Revenue|
+---------+-------+
|Hyderabad|   5000|
|    Delhi|   6000|
|   Mumbai|   1500|
|Bangalore|  13000|
|  Chennai|   7000|
|  Kolkata|   9000|
|Ahmedabad|   5000|
+---------+-------+



In [0]:
spark.sql("select * from patients order by bill_amount desc limit 1").show()

+--------+------------+---------+----------+----------------+-----------+-----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|bill_amount|
+--------+------------+---------+----------+----------------+-----------+-----------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      10000|
+--------+------------+---------+----------+----------------+-----------+-----------+



In [0]:
spark.sql("select department,count(*) Count from patients group by department").show()

+-----------+-----+
| department|Count|
+-----------+-----+
| Cardiology|    3|
|Orthopedics|    2|
|Dermatology|    2|
|  Neurology|    1|
+-----------+-----+



---

## 🔷 PART 3 — DELTA LAKE (CORE OPERATIONS)

### Tasks

1. Create a Delta table from the dataset
2. Insert new records
3. Update existing records
4. Delete specific records
5. Perform an UPSERT using MERGE

---

In [0]:
df.write.format("delta").saveAsTable("patients_delta")

In [0]:
%sql
INSERT INTO patients_delta VALUES
(109,'Preethi','Chennai','Cardiology',6000,2,12000),
(110,'Rohit Mehta','Mumbai','Neurology',6500,2,13000),
(111,'Divya Sharma','Delhi','Cardiology',5000,3,15000),
(112,'Sanjay Gupta','Kolkata','Orthopedics',3000,2,6000),
(113,'Pooja Verma','Bangalore','Dermatology',2000,2,4000),
(114,'Amit Patel','Ahmedabad','Cardiology',5500,1,5500),
(115,'Neha Reddy','Hyderabad','Neurology',7000,2,14000),
(116,'Karthik Subramanian','Chennai','Orthopedics',3500,3,10500),
(117,'Anjali Nair','Bangalore','Cardiology',5000,2,10000),
(118,'Vishal Singh','Delhi','Dermatology',1800,2,3600),
(119,'Sneha Iyer','Chennai','Neurology',7200,1,7200);

num_affected_rows,num_inserted_rows
11,11


In [0]:
%sql
update patients_delta set consultation_fee=7000 where department="Neurology";

num_affected_rows
4


In [0]:
%sql
delete from patients_delta where tests_count>2;

num_affected_rows
3


In [0]:
%sql

create or replace temp view new_patients as
select * from values
(101,'Arjun Reddy','Hyderabad','Cardiology',8000,2,16000),   
(111,'Divya Sharma','Delhi','Cardiology',5000,3,15000),      
(112,'Sanjay Gupta','Kolkata','Orthopedics',3000,2,6000),    
(105,'Vikram Singh','Chennai','Neurology',7500,1,7500)       
as new_patients(visit_id,patient_name,city,department,consultation_fee,tests_count,bill_amount);

In [0]:
%sql

merge into patients_delta as target
using new_patients as source
on target.visit_id = source.visit_id
when matched then
update set
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.bill_amount = source.bill_amount
when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,3,0,1


---

## 🔷 PART 4 — DELTA ADVANCED

### Tasks

1. Retrieve table history
2. Query an older version of the table
3. Explain the effect of VACUUM
4. Execute VACUUM (dry run)

---

In [0]:
%sql
describe history patients_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-05-04T04:56:17.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3665098557635061),8be6dfbd-23f1-4671-ac9d-002edd13affc,0504-040620-uxwkmy10-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 10925, p25FileSize -> 2747, numDeletionVectorsRemoved -> 1, minFileSize -> 2747, numAddedFiles -> 1, maxFileSize -> 2747, p75FileSize -> 2747, p50FileSize -> 2747, numAddedBytes -> 2747)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T04:56:15.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#15073L = cast(visit_id#15059 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3665098557635061),8be6dfbd-23f1-4671-ac9d-002edd13affc,0504-040620-uxwkmy10-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 8240, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 4061, materializeSourceTimeMs -> 357, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1545, numTargetRowsUpdated -> 3, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2099)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T04:40:35.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3665098557635061),1cca3dee-13a8-474a-9ea9-689c17dbb12a,0504-040620-uxwkmy10-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2763, p25FileSize -> 2685, numDeletionVectorsRemoved -> 1, minFileSize -> 2685, numAddedFiles -> 1, maxFileSize -> 2685, p75FileSize -> 2685, p50FileSize -> 2685, numAddedBytes -> 2685)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T04:40:34.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(tests_count#14606L > 2)""])",null,List(3665098557635061),1cca3dee-13a8-474a-9ea9-689c17dbb12a,0504-040620-uxwkmy10-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1516, numDeletionVectorsUpdated -> 0, numDeletedRows -> 3, scanTimeMs -> 1049, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 466)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T04:39:17.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3665098557635061),8d13c183-54d6-4c8c-8d44-d2f10fb4f1a8,0504-040620-uxwkmy10-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7432, p25FileSize -> 2763, numDeletionVectorsRemoved -> 2, minFileSize -> 2763, numAddedFiles -> 1, maxFileSize -> 2763, p75FileSize -> 2763, p50FileSize -> 2763, numAddedBytes -> 2763)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T04:39:15.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(department#14046 = Neurology)""])",null,List(3665098557635061),8d13c183-54d6-4

In [0]:
%sql
SELECT * FROM patients_delta VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,bill_amount
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,6000
103,Rahul Sharma,Mumbai,Dermatology,1500,1,1500
104,Priya Nair,Bangalore,Cardiology,5000,2,10000
105,Vikram Singh,Chennai,Neurology,7000,1,7000
106,Ananya Das,Kolkata,Orthopedics,3000,3,9000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5000
108,Meera Iyer,Bangalore,Dermatology,1500,2,3000


In [0]:
%sql
VACUUM patients_delta RETAIN 168 HOURS DRY RUN;

path


---

## 🔷 PART 6 — INCREMENTAL LOAD

### Scenario

You receive daily updates.

### Tasks

1. Create a target Delta table
2. Create a dataset representing daily updates
3. Implement incremental load logic
4. Ensure:

   * Existing records are updated
   * New records are inserted

---

In [0]:
%sql

create or replace table target_table
using delta as
select * from values
(201,'Aditya Rao','Hyderabad','Cardiology',5200,2,10400),
(202,'Nisha Verma','Delhi','Dermatology',1800,3,5400),
(203,'Rakesh Pillai','Kochi','Neurology',7500,1,7500),
(204,'Sneha Kulkarni','Pune','Orthopedics',3200,2,6400),
(205,'Arvind Menon','Chennai','Cardiology',6000,1,6000),
(206,'Pallavi Shah','Ahmedabad','Dermatology',2000,2,4000),
(207,'Kunal Singh','Lucknow','Orthopedics',3500,3,10500),
(208,'Megha Joshi','Jaipur','Neurology',7000,2,14000),
(209,'Siddharth Jain','Indore','Cardiology',5000,2,10000),
(210,'Anita Das','Kolkata','Dermatology',1500,4,6000)    
as target_table(visit_id,patient_name,city,department,consultation_fee,tests_count,bill_amount);

num_affected_rows,num_inserted_rows


In [0]:
%sql
create or replace temp view new_patients_updates as
select * from values
(201,'Aditya Rao','Hyderabad','Cardiology',5500,2,11000),   
(202,'Nisha Verma','Delhi','Dermatology',2000,3,6000),      
(211,'Rahul Khanna','Mumbai','Neurology',7200,1,7200),      
(212,'Priya Nair','Bangalore','Orthopedics',3300,2,6600),   
(205,'Arvind Menon','Chennai','Cardiology',6500,1,6500)     
as new_patients_updates(visit_id,patient_name,city,department,consultation_fee,tests_count,bill_amount);

In [0]:
%sql

merge into target_table as target
using new_patients_updates as source
on target.visit_id = source.visit_id
when matched then
update set
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.bill_amount = source.bill_amount
when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2



---

## 🔷 PART 8 — UNITY CATALOG

### Tasks

1. Create a catalog
2. Create a schema
3. Create a table inside schema
4. Load data into the table

---


In [0]:
%sql
create catalog hospital_patients_catelog;

In [0]:
%sql
use catalog hospital_patients_catelog;

In [0]:
%sql
create schema patient_schema;

In [0]:
%sql
create table patient_schema.patients_uc using delta as select * from patients;

num_affected_rows,num_inserted_rows


---

## 🔷 PART 9 — DATA GOVERNANCE

### Tasks

1. Discover tables using catalog
2. Create a derived table
3. Observe lineage
4. Apply access control
5. Query audit logs

---

In [0]:
%sql
show tables in patient_schema;

database,tableName,isTemporary
patient_schema,patients_uc,false
,new_patients,true
,new_patients_updates,true
,patients,true


In [0]:
%sql
create table patient_schema.high_value as
select * from patient_schema.patients_uc where bill_amount > 5000;

num_affected_rows,num_inserted_rows


In [0]:
%sql
GRANT SELECT ON TABLE patient_schema.patients_uc 
TO `azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com`;

In [0]:
%sql
SELECT * FROM patient_schema.patients_uc;

INSERT INTO patient_schema.patients_uc VALUES
(300,'Test User','Chennai','Cardiology',5000,1,5000);

UPDATE patient_schema.patients_uc
SET consultation_fee = 6000
WHERE visit_id = 300;

DELETE FROM patient_schema.patients_uc
WHERE visit_id = 300;

num_affected_rows
1


In [0]:
%sql
DESCRIBE HISTORY patient_schema.patients_uc;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-05-04T05:55:49.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(visit_id#20349L = 300)""])",null,List(3665098557635061),a223fe8e-e872-4759-8c30-3a577a62783b,0504-040620-uxwkmy10-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2084, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 945, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 849, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 96)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T05:55:47.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_id#19957L = 300)""])",null,List(3665098557635061),b4024d87-bd74-4419-92f2-006e6a095676,0504-040620-uxwkmy10-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2040, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1807, numDeletionVectorsUpdated -> 0, scanTimeMs -> 880, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2084, rewriteTimeMs -> 926)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-05-04T05:55:45.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3665098557635061),fb5115a3-d4a0-4cb2-9c8d-5226eb182fd3,0504-040620-uxwkmy10-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 2040)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-04T05:43:37.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3665098557635061),fdd622e2-ceea-4659-a1dd-060e4e8f9808,0504-040620-uxwkmy10-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 8, numOutputBytes -> 2558)",null,Databricks-Runtime/18.1.x-photon-scala2.13
